# Train ResNet-SE + MixUp + EMA CNN on Colab GPU — Run All

Reproduces **exactly** the local pipeline in `CNN/cnn_resnet_se_mixup_ema/` — same
`train.py` / `test.py`, same flags, same output files — just on a Colab GPU
instead of your Mac's MPS, so results are 1:1 with a local run (only wall-clock
differs).

## One-time setup (outside this notebook, before you hit Run All)

`drive.mount()` does not work through the VS Code↔Colab bridge (Google's auth
tunnel rejects it with a 400 — a bridge limitation, not something fixable by
retrying), so this version skips Drive's mount entirely and downloads the two
zips with `gdown`, which needs no interactive auth at all.

1. On your Mac, from the repo root:
   ```bash
   zip -r emotion-recognition-code.zip . -x 'data/*' '*/checkpoints/*.pt' '.venv/*' '.git/*'
   zip -r data.zip data
   ```
2. Upload both zips to Google Drive, then for **each file**: right-click ->
   Share -> "General access" -> **Anyone with the link** -> Copy link.
   The link looks like `https://drive.google.com/file/d/FILE_ID/view?usp=sharing` —
   you only need the `FILE_ID` part (the long string between `/d/` and `/view`).
3. Paste the two `FILE_ID`s into the `CODE_FILE_ID` / `DATA_FILE_ID` variables in
   the next cell.
4. In VS Code: `Cmd+Shift+P` -> "Colab" -> connect this notebook to a Colab
   runtime with a **GPU** (T4 is enough).
5. **Run All** — no auth popups, runs unattended (~15-25 min).

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU on this runtime — reconnect with a GPU accelerator (T4) before Run All."
print("GPU:", torch.cuda.get_device_name(0))
print("xd") 

In [ ]:
# --- Paste your two Google Drive file IDs here (see setup step 2-3 above) ---
CODE_FILE_ID = "1xKUBxcENttg17d__d_bRq_dnHn20fpQW"
DATA_FILE_ID = "1PuTuPogeHphWy3oFeJReW5bswA9KtBt0"

%pip install -q gdown

import os, zipfile

# Don't assume /content — the VS Code<->Colab bridge doesn't always land you
# there. Use wherever this kernel's cwd actually is.
WORK_DIR = os.path.join(os.getcwd(), "emotion-recognition")
os.makedirs(WORK_DIR, exist_ok=True)
print("Working directory will be:", WORK_DIR)

code_zip = os.path.join(os.getcwd(), "emotion-recognition-code.zip")
data_zip = os.path.join(os.getcwd(), "data.zip")

!gdown {CODE_FILE_ID} -O {code_zip}
!gdown {DATA_FILE_ID} -O {data_zip}

assert os.path.exists(code_zip), f"Download failed — {code_zip} not found. Check CODE_FILE_ID and sharing permissions."
assert os.path.exists(data_zip), f"Download failed — {data_zip} not found. Check DATA_FILE_ID and sharing permissions."

with zipfile.ZipFile(code_zip, "r") as z:
    z.extractall(WORK_DIR)

%cd {WORK_DIR}

with zipfile.ZipFile(data_zip, "r") as z:
    z.extractall(".")  # -> ./data/train, ./data/test

print("Code + data ready in:", os.getcwd())
!ls
!ls data

In [ ]:
# Sanity check: same class folders / rough counts as local.
for split in ["train", "test"]:
    for cls in sorted(os.listdir(f"data/{split}")):
        n = len(os.listdir(f"data/{split}/{cls}"))
        print(f"{split}/{cls}: {n}")

In [ ]:
# Colab ships torch/torchvision (CUDA build) already; just add the rest.
%pip install -q scikit-learn matplotlib
import torchvision, sklearn
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| cuda", torch.cuda.is_available())

## Train (~15-20 min on a T4, 60 epochs — identical recipe to local)

In [ ]:
!python -m CNN.cnn_resnet_se_mixup_ema.train 2>&1 | tee CNN/cnn_resnet_se_mixup_ema/logs.txt

## Test — no TTA, then multi-crop TTA (headline numbers)

In [ ]:
!python -m CNN.cnn_resnet_se_mixup_ema.test 2>&1 | tee CNN/cnn_resnet_se_mixup_ema/test_no_tta.log
# Keep a copy before the next cell's --tta-multi run overwrites metrics.json/.txt.
!cp CNN/cnn_resnet_se_mixup_ema/results/metrics.json CNN/cnn_resnet_se_mixup_ema/results/metrics_no_tta.json
!cp CNN/cnn_resnet_se_mixup_ema/results/metrics.txt CNN/cnn_resnet_se_mixup_ema/results/metrics_no_tta.txt

In [ ]:
!python -m CNN.cnn_resnet_se_mixup_ema.test --tta-multi 2>&1 | tee CNN/cnn_resnet_se_mixup_ema/test_tta_multi.log

## Auto-fill RESULTS.md from the real numbers (same template you'd fill by hand locally)

In [ ]:
import json, re

RESULTS_PATH = "CNN/cnn_resnet_se_mixup_ema/RESULTS.md"

with open("CNN/cnn_resnet_se_mixup_ema/results/history.json") as f:
    history = json.load(f)
with open("CNN/cnn_resnet_se_mixup_ema/results/metrics_no_tta.json") as f:
    no_tta = json.load(f)
with open("CNN/cnn_resnet_se_mixup_ema/results/metrics.json") as f:
    tta = json.load(f)  # multi-TTA run (last one executed)
with open("CNN/cnn_resnet_se_mixup_ema/logs.txt") as f:
    log_text = f.read()

best_val_acc = max(history["val_acc"])
best_epoch = history["val_acc"].index(best_val_acc) + 1

m_flops = re.search(r"Total training compute: ([\d.e+]+) FLOPs \((\d+) samples seen\)", log_text)
total_flops = m_flops.group(1) if m_flops else "???"
samples_seen = m_flops.group(2) if m_flops else "???"

classes = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

md = open(RESULTS_PATH).read()

status_old = md.split("> **STATUS: PENDING TRAINING RUN.**")[1].split("\n\n")[0]
md = md.replace(
    "> **STATUS: PENDING TRAINING RUN.**" + status_old + "\n\n",
    "> **STATUS: DONE.** Trained on a Colab GPU runtime via `train_colab.ipynb` "
    "(Run All); all numbers below are real measured results, filled in "
    "automatically from `results/metrics.json`, `results/metrics_no_tta.json`, "
    "and `logs.txt`.\n\n",
)
md = md.replace(
    "Checkpoint: `checkpoints/best.pt` (EMA weights, epoch `___`/60, val_acc `___`).",
    f"Checkpoint: `checkpoints/best.pt` (EMA weights, epoch {best_epoch}/60, val_acc {best_val_acc:.4f}).",
)

md = md.replace(
    "| Accuracy    | 0.6648 | 0.6956 | `___` | `___` |",
    f"| Accuracy    | 0.6648 | 0.6956 | {no_tta['accuracy']:.4f} | {tta['accuracy']:.4f} |",
)
md = md.replace(
    "| Macro F1    | 0.6469 | 0.6611 | `___` | `___` |",
    f"| Macro F1    | 0.6469 | 0.6611 | {no_tta['macro_f1']:.4f} | {tta['macro_f1']:.4f} |",
)
md = md.replace(
    "| Weighted F1 | 0.6625 | 0.6956 | `___` | `___` |",
    f"| Weighted F1 | 0.6625 | 0.6956 | {no_tta['weighted_f1']:.4f} | {tta['weighted_f1']:.4f} |",
)
md = md.replace(
    "Best validation accuracy (EMA): `___` (baseline 0.6655, GAP/aug/LS 0.6864).",
    f"Best validation accuracy (EMA): {best_val_acc:.4f} (baseline 0.6655, GAP/aug/LS 0.6864).",
)

if samples_seen != "???":
    md = md.replace(
        "| Samples seen | 1,034,240 | 1,033,560 | \u2248 2,583,900 |",
        f"| Samples seen | 1,034,240 | 1,033,560 | {int(samples_seen):,} |",
    )
md = md.replace(
    "| **Total training FLOPs** | 2.13e15 | 2.91e15 | **\u2248 1.93e16** |",
    f"| **Total training FLOPs** | 2.13e15 | 2.91e15 | **{total_flops}** |",
)
md = md.replace(
    "| Approx. wall-clock (MPS) | ~36 min | ~45 min | **~3.7 h** |",
    "| Approx. wall-clock | ~36 min (MPS) | ~45 min (MPS) | **see logs.txt (Colab T4)** |",
)

for cls in classes:
    row = tta["per_class"][cls]
    old_line = next(line for line in md.split("\n") if line.strip().startswith(f"| {cls}"))
    new_line = old_line.replace(
        "`___` | `___` | `___`",
        f"{row['precision']:.4f} | {row['recall']:.4f} | {row['f1-score']:.4f}",
    )
    new_line = re.sub(r"\| `___`\s*$", f"| {row['f1-score']:.4f}", new_line)
    md = md.replace(old_line, new_line)

with open(RESULTS_PATH, "w") as f:
    f.write(md)

print("RESULTS.md filled in.")
print(f"Best val acc: {best_val_acc:.4f} @ epoch {best_epoch}")
print(f"Test acc -> no TTA: {no_tta['accuracy']:.4f} | multi-TTA: {tta['accuracy']:.4f}")

## Package results for download

Drive mount doesn't work through the VS Code bridge, so this just zips
everything into one file in the current working directory. **Grab it via
VS Code's Colab file browser:** open the file explorer panel for this runtime,
find `cnn_resnet_se_mixup_ema_results.zip` (the exact path is printed by the
cell below — it's wherever this notebook's cwd ended up, not necessarily
`/content`), right-click -> Download. Then unzip locally and drop the contents
into `CNN/cnn_resnet_se_mixup_ema/`, overwriting the placeholder files.

In [ ]:
import os

!zip -r cnn_resnet_se_mixup_ema_results.zip \
    CNN/cnn_resnet_se_mixup_ema/checkpoints/best.pt \
    CNN/cnn_resnet_se_mixup_ema/results \
    CNN/cnn_resnet_se_mixup_ema/logs.txt \
    CNN/cnn_resnet_se_mixup_ema/test_no_tta.log \
    CNN/cnn_resnet_se_mixup_ema/test_tta_multi.log \
    CNN/cnn_resnet_se_mixup_ema/RESULTS.md

result_zip_path = os.path.join(os.getcwd(), "cnn_resnet_se_mixup_ema_results.zip")
print(f"Packaged: {result_zip_path}")
print("Download it via the VS Code Colab file browser (right-click -> Download),")
print("then unzip into CNN/cnn_resnet_se_mixup_ema/ locally.")